# Verso Fine-Tuning — Qwen 2.5 3B with Unsloth

Fine-tunes Qwen 2.5 3B to generate learning reels + flashcards as JSON.

**RunPod setup:** RTX 3090 or RTX 4090, RunPod PyTorch template

**Steps:**
1. Install Unsloth
2. Load model (4-bit)
3. Apply LoRA
4. Train on `verso_training_sharegpt.json`
5. Export GGUF for Ollama

In [ ]:
# Cell 1: Install Unsloth
!pip install --no-deps unsloth
!pip install transformers datasets trl peft accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# Cell 2: Upload your training data
# Upload verso_training_sharegpt.json to /workspace/ before running this cell

import json
import os

DATA_PATH = "/workspace/verso_training_sharegpt.json"
assert os.path.exists(DATA_PATH), f"Upload {DATA_PATH} first!"

with open(DATA_PATH) as f:
    data = json.load(f)
print(f"Loaded {len(data)} training examples")
print(f"Sample keys: {list(data[0].keys())}")

In [ ]:
# Cell 3: Load Qwen 2.5 3B Instruct with 4-bit quantization
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Cell 4: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # memory efficient
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")

In [ ]:
# Cell 5: Prepare dataset in ShareGPT/chat format
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Load and standardize
dataset = Dataset.from_list(data)
dataset = standardize_sharegpt(dataset)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print(f"Dataset ready: {len(dataset)} examples")
print(f"\nSample (first 500 chars):\n{dataset[0]['text'][:500]}")

In [ ]:
# Cell 6: Configure trainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # effective batch size = 16
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./verso_output",
        report_to="none",
    ),
)

print("Trainer configured. Ready to train.")

In [ ]:
# Cell 7: Train!
import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final loss: {stats.training_loss:.4f}")

In [ ]:
# Cell 8: Quick test — generate a sample output
FastLanguageModel.for_inference(model)

test_prompt = """Document type: textbook
Learning style: visual
Depth: balanced
Focus: learning
Difficulty: medium

Text:
Photosynthesis is the process by which green plants convert light energy into chemical energy. It occurs in chloroplasts and involves two stages: light reactions and the Calvin cycle.

JSON:"""

messages = [
    {"role": "system", "content": "You are Verso, a learning content creator. Generate reels and flashcards as JSON."},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
output = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.3)
response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== Test Output ===")
print(response[:1000])

In [ ]:
# Cell 9: Export to GGUF for Ollama
# Q4_K_M is the sweet spot for quality vs size on 8GB RAM

model.save_pretrained_gguf(
    "verso-qwen2.5-3b",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nGGUF export complete!")
print("File: verso-qwen2.5-3b/verso-qwen2.5-3b.Q4_K_M.gguf")

import os
gguf_path = "verso-qwen2.5-3b/verso-qwen2.5-3b.Q4_K_M.gguf"
if os.path.exists(gguf_path):
    size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
    print(f"Size: {size_mb:.0f} MB")
else:
    # Unsloth may use a different naming convention
    for f in os.listdir("verso-qwen2.5-3b"):
        if f.endswith(".gguf"):
            size_mb = os.path.getsize(f"verso-qwen2.5-3b/{f}") / (1024*1024)
            print(f"Found: {f} ({size_mb:.0f} MB)")

## Next Steps

1. **Download** the `.gguf` file from `verso-qwen2.5-3b/`
2. **Copy** it to your local machine alongside `scripts/Modelfile`
3. **Create** the Ollama model locally:
   ```bash
   cd scripts
   ollama create verso-model -f Modelfile
   ```
4. **Update** `backend/config.py`: `LLM_MODEL = "verso-model"`
5. **Re-run evals** to compare against 89.2% baseline:
   ```bash
   docker compose exec backend python evals.py
   ```